In [18]:
# DiD

import pandas as pd
import statsmodels.formula.api as smf


# Load dataset

df = pd.read_stata("brc_core.dta")

# Filter sample (sample3 == 1)

if 'sample3' in df.columns:
    df = df[df['sample3'] == 1].copy()


# Drop rows with missing key variables

key_cols = ['cites9', 'brc', 'year', 'pub_year', 'pair_num'] + [f'age{i}' for i in range(1,31)]
df = df.dropna(subset=key_cols)
df['pair_num'] = df['pair_num'].astype(int)  # ensure integer for clustering


# Create window and post-deposit dummies

df['window'] = ((df['year'] >= df['pub_year'] - 1) & (df['year'] <= df['pub_year'] + 1)).astype(int)
df['post_brc'] = (df['year'] >= df['pub_year'] + 2).astype(int)


# Create interaction terms

df['brc_window'] = df['brc'] * df['window']
df['brc_post'] = df['brc'] * df['post_brc']

# -----------------------------
# 6. Build regression formula
# -----------------------------
age_cols = [f'age{i}' for i in range(1,31)]
formula = 'cites9 ~ brc_window + brc_post'
formula += ' + ' + ' + '.join(age_cols)


# Run OLS with cluster-robust SE

model = smf.ols(formula=formula, data=df).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['pair_num']}
)


# Display results

print(model.summary())

# Print DiD estimates and standard errors

print("DiD estimates:")
print("Window period effect (brc_window):", model.params['brc_window'])

print("\nStandard errors:")
print("Window period effect SE:", model.bse['brc_window'])

beta_window = model.params['brc_window']
beta_post = model.params['brc_post']


# 9. Compute % increase relative to controls

# Window period

control_window_mean = df[(df['window']==1) & (df['brc']==0)]['cites9'].mean()
pct_window = (beta_window / control_window_mean) * 100

# Post-deposit period

control_post_mean = df[(df['post_brc']==1) & (df['brc']==0)]['cites9'].mean()
pct_post = (beta_post / control_post_mean) * 100

print("\nDiD % increases (OLS):")
print(f"Window-period effect: {pct_window:.1f}%")
print(f"Post-deposit effect: {pct_post:.1f}%")


                            OLS Regression Results                            
Dep. Variable:                 cites9   R-squared:                       0.239
Model:                            OLS   Adj. R-squared:                  0.234
Method:                 Least Squares   F-statistic:                     17.82
Date:                Thu, 18 Dec 2025   Prob (F-statistic):           5.34e-30
Time:                        21:32:51   Log-Likelihood:                -7110.8
No. Observations:                4857   AIC:                         1.429e+04
Df Residuals:                    4824   BIC:                         1.450e+04
Df Model:                          32                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2209      0.067      3.309      0.0

In [15]:
# Cross moment

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression


# Load dataset and filter sample

df = pd.read_stata("brc_core.dta")
if 'sample3' in df.columns:
    df = df[df['sample3'] == 1].copy()

# Drop missing key variables

key_cols = ['cites9', 'brc', 'year', 'pub_year', 'rart_num'] + [f'age{i}' for i in range(1,31)]
df = df.dropna(subset=key_cols)

# Covariates

covariates = [f'age{i}' for i in range(1,31)]


# Create window and post-deposit dummies

df['window'] = ((df['year'] >= df['pub_year'] - 1) & (df['year'] <= df['pub_year'] + 1)).astype(int)
df['post_brc'] = (df['year'] >= df['pub_year'] + 2).astype(int)


# Split pre/post periods

df_pre = df[df['window'] == 1].copy()
df_post = df[df['post_brc'] == 1].copy()


# 4. Aggregate outcomes per article

# Average citations per article

df_pre_agg = df_pre.groupby('rart_num')['cites9'].mean().reset_index()
df_post_agg = df_post.groupby('rart_num')['cites9'].mean().reset_index()

# Take first row for covariates (age, brc)

cov_pre = df_pre.groupby('rart_num')[covariates + ['brc']].first().reset_index()
cov_post = df_post.groupby('rart_num')[covariates].first().reset_index()

# Merge aggregated outcomes with covariates

df_pre_matched = pd.merge(df_pre_agg, cov_pre, on='rart_num')
df_post_matched = pd.merge(df_post_agg, cov_post, on='rart_num')

# Keep only articles present in both periods

common_ids = set(df_pre_matched['rart_num']).intersection(df_post_matched['rart_num'])
df_pre_matched = df_pre_matched[df_pre_matched['rart_num'].isin(common_ids)].sort_values('rart_num')
df_post_matched = df_post_matched[df_post_matched['rart_num'].isin(common_ids)].sort_values('rart_num')


# Extract outcomes, covariates, and treatment

Y_pre = df_pre_matched['cites9'].to_numpy()
Y_post = df_post_matched['cites9'].to_numpy()
X_pre = df_pre_matched[covariates].to_numpy()
X_post = df_post_matched[covariates].to_numpy()
D_pre = df_pre_matched['brc'].to_numpy().astype(float)


# Residualize outcomes on covariates

reg = LinearRegression().fit(np.vstack([X_pre, X_post]), np.concatenate([Y_pre, Y_post]))
Z = Y_pre - np.matmul(X_pre, reg.coef_)
Y_post_resid = Y_post - np.matmul(X_post, reg.coef_)

# Center variables

Z -= np.mean(Z)
D_pre -= np.mean(D_pre)
Y_post_resid -= np.mean(Y_post_resid)


# Cross-moment functions

def compute_ratio(Z, D, deg=2):
    var_u = np.mean(Z * D)
    sign = np.sign(var_u)
    diff_normal_D = np.mean(D ** deg * Z) - deg * var_u * np.mean(D ** (deg - 1))
    diff_normal_Z = np.mean(Z ** deg * D) - deg * var_u * np.mean(Z ** (deg - 1))
    epsilon = 1e-8
    alpha_sq = diff_normal_D / (diff_normal_Z + epsilon)
    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq) ** (1 / (deg - 1)))
    else:
        alpha_sq = alpha_sq ** (1 / (deg - 1))
    alpha_sq = abs(alpha_sq) * sign
    return alpha_sq

def get_beta(Z, D, Y, deg=2):
    denominator = 0
    while denominator == 0:
        alpha_sq = compute_ratio(Z, D, deg)
        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)
        deg += 1
    return numerator / denominator


# Estimate cross-moment beta

max_deg = 12
betas_est = np.zeros(max_deg - 2)

for deg in range(2, max_deg):
    beta = get_beta(Z, D_pre, Y_post_resid, deg)
    betas_est[deg - 2] = beta

print("===== Cross-Moment Beta Estimation =====")
print("Beta estimate (deg=2):", betas_est[0])


===== Cross-Moment Beta Estimation =====
Beta estimate (deg=2): 0.9149317628865606
